# Setup


In [ ]:
%pip install -q -U "crewai[google-genai]"

# Auth & Flow


In [ ]:
import os
import json
from google.colab import userdata

from crewai import Agent, Task, Crew, LLM
from crewai.flow.flow import Flow, start, listen, router, or_
from pydantic import BaseModel, Field

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "Add GEMINI_API_KEY in Google Colab Secrets."
    )

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY

llm = LLM(
    model="gemini/gemini-3.5-flash-lite",
    api_key=GEMINI_API_KEY,
    temperature=0.1
)

print("Gemini configured successfully!")

# Issue state


In [ ]:
class BugTriageState(BaseModel):
    issue_title: str = ""
    issue_body: str = ""
    issue_type: str = "question"
    classification_reason: str = ""
    result: dict = Field(default_factory=dict)

# Sample issue


In [ ]:
sample_issue = {
    "title": "Session token expires during checkout without preserving form draft",
    "body": """
    When users take longer than 15 minutes to complete the checkout form,
    the session expires and redirects to login, wiping all entered address
    and order fields. Tested on both Chrome and Firefox.
    Smaller single-item orders succeed normally without delay.
    """
}

print(sample_issue["title"])

# Triage flow


In [ ]:
class BugTriageFlow(Flow[BugTriageState]):

    @start()
    def read_issue(self):
        print("\n📥 Reading issue...")

        self.state.issue_title = sample_issue["title"]
        self.state.issue_body = sample_issue["body"]

        print("Title:", self.state.issue_title)
        return "issue_loaded"

    @listen(read_issue)
    def classify_issue(self, _):
        print("\n🔍 Classifying issue...")

        prompt = f"""
Classify the following software issue into exactly one category:
bug, feature, or question.

Title:
{self.state.issue_title}

Body:
{self.state.issue_body}

Rules:
- bug: existing behavior is broken, incorrect, or crashes.
- feature: requests new or enhanced functionality.
- question: asks for information or help without reporting
  a software defect or requesting a feature.

Return ONLY valid JSON:
{{
  "issue_type": "bug",
  "reason": "Brief explanation"
}}
"""

        response = llm.call(prompt)

        try:
            raw = str(response).strip()
            raw = raw.replace("```json", "").replace("```", "").strip()
            data = json.loads(raw)

            issue_type = str(data["issue_type"]).lower().strip()

            if issue_type not in {"bug", "feature", "question"}:
                raise ValueError("Invalid issue category")

            self.state.issue_type = issue_type
            self.state.classification_reason = str(data["reason"])

        except (ValueError, TypeError, KeyError, json.JSONDecodeError):
            self.state.issue_type = "question"
            self.state.classification_reason = (
                "Classification could not be parsed; "
                "manual review is required."
            )

        print("Category:", self.state.issue_type)
        print("Reason:", self.state.classification_reason)

        return self.state.issue_type

    @router(classify_issue)
    def route_type(self):
        category = self.state.issue_type

        print(f"\n➡️ Routing to: {category}")

        if category == "bug":
            return "bug"
        elif category == "feature":
            return "feature"
        else:
            return "question"

    @listen("bug")
    def bug_crew(self):
        agent = Agent(
            role="Defect Triage Specialist",
            goal="Investigate reported defects and provide structured reproduction steps",
            backstory=(
                "You help software teams understand defects, "
                "severity, and steps to reproduce them."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Analyze this reported bug.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

Classification reason: {self.state.classification_reason}

Provide:
1. Suggested severity: low, medium, high, or critical
2. Steps to reproduce based on the supplied information
3. Expected versus actual behavior
4. Missing information that needs verification

Do not invent technical details.
""",
            expected_output="A structured bug triage report.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "bug",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n🐛 Bug analysis completed.")
        return "bug_completed"

    @listen("feature")
    def feature_crew(self):
        agent = Agent(
            role="Feature Specification Analyst",
            goal="Translate enhancement proposals into structured user stories",
            backstory=(
                "You help product teams define user needs "
                "and acceptance criteria."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Turn this feature request into a draft user story.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

Include:
1. User story in the format As a..., I want..., so that...
2. Acceptance criteria
3. Clarifying questions

Do not invent requirements that are not supported by the issue.
""",
            expected_output="A user story with acceptance criteria.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "feature",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n💡 Feature story drafted.")
        return "feature_completed"

    @listen("question")
    def answer_crew(self):
        agent = Agent(
            role="Technical Support Assistant",
            goal="Draft a clear, accurate reply addressing the technical question",
            backstory=(
                "You help users understand software behavior "
                "without inventing undocumented facts."
            ),
            llm=llm,
            verbose=False
        )

        task = Task(
            description=f"""
Draft a concise, helpful response to this issue.

Title: {self.state.issue_title}
Body: {self.state.issue_body}

If documentation or technical details are missing, clearly
state what information is needed. Do not invent product facts.
""",
            expected_output="A clear draft support response.",
            agent=agent
        )

        result = str(Crew(
            agents=[agent],
            tasks=[task],
            verbose=False
        ).kickoff())

        self.state.result = {
            "category": "question",
            "title": self.state.issue_title,
            "analysis": result
        }

        print("\n💬 Support response drafted.")
        return "question_completed"

    @listen(or_(bug_crew, feature_crew, answer_crew))
    def post_reply(self, _):
        print("\n📌 Preparing issue response...")

        self.state.result["classification_reason"] = (
            self.state.classification_reason
        )
        self.state.result["reply_status"] = "Draft ready for human review"

        output_path = "/content/bug_triage_result.json"

        with open(output_path, "w", encoding="utf-8") as file:
            json.dump(
                self.state.result,
                file,
                indent=2,
                ensure_ascii=False
            )

        print("Category:", self.state.result["category"])
        print("Saved:", output_path)
        print("Reply prepared for human review.")

        return output_path

# Run


In [ ]:
flow = BugTriageFlow()
flow.kickoff()

# Output


In [ ]:
print(json.dumps(flow.state.result, indent=2, ensure_ascii=False))

# Download


In [ ]:
from google.colab import files

files.download("/content/bug_triage_result.json")